In [361]:
# basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [362]:
df_1 = pd.read_csv("scraped_steam_games.csv")
df_1.head()
#each csv is around 500 games long, will have to concatenate into one big dataframe


,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s)
0,Counter-Strike 2,"Aug 21, 2012",Free To Play,No Discount,Valve,Valve,"9,767,142","8,401,470","1,365,672",Mostly Positive,"['Action', 'Free To Play']"
1,Palworld,"Jul 9, 2026",$29.99,No Discount,Pocketpair,Pocketpair,"466,128","440,315","25,813",Very Positive,"['Action', 'Adventure', 'Indie', 'RPG']"
2,Baldur's Gate 3,"Aug 3, 2023",$59.99,$41.99,Larian Studios,Larian Studios,"850,775","823,465","27,310",Very Positive,"['Adventure', 'RPG', 'Strategy']"
3,Marvel Rivals,"Dec 5, 2024",Free To Play,No Discount,NetEase Games,NetEase Games,"405,407","307,754","97,653",Mixed,"['Action', 'Free To Play']"
4,Warframe,"Mar 25, 2013",$1.99,$1.59,Digital Extremes,Digital Extremes,"678,882","593,641","85,241",Very Positive,"['Action', 'RPG', 'Free To Play']"


In [363]:
df_2 = pd.read_csv("scraped_steam_games_2.csv")
df_2.head()

,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s)
0,shapez 2 - Factory,"Apr 23, 2026",$29.99,$17.99,tobspr Games,tobspr Games,"14,985","14,546",439,Very Positive,"['Casual', 'Indie', 'Simulation', 'Strategy']"
1,Blue Archive,"Jul 3, 2025",Free To Play,No Discount,NEXON Games,NEXON Korea,"27,467","17,836","9,631",Overwhelmingly Negative,"['Adventure', 'Casual', 'RPG', 'Strategy']"
2,DRAGON BALL GEKISHIN SQUADRA,"Sep 9, 2025",Free To Play,No Discount,"GANBARION Co., Ltd.",Bandai Namco Entertainment,"10,606","6,904","3,702",Mixed,"['Action', 'Free To Play']"
3,Infection Free Zone,"Apr 11, 2024",$44.98,$17.31,Jutsu Games,Games Operators,"18,349","14,784","3,565",Very Positive,"['Action', 'Indie', 'Simulation', 'Strategy']"
4,Soul Land: Awakening World,"Jun 1, 2026",Free To Play,No Discount,37GAMES,37GAMES,348,167,181,Mixed,"['Action', 'Adventure', 'Massively Multiplayer..."


In [364]:
#concat the two dataframes together
df = pd.concat([df_1, df_2])
print(df.shape)

(996, 11)


# Data Cleaning

In [365]:
#go feature by feature
#date parse
df["Release Date"] = pd.to_datetime(df["Release Date"], format = "mixed")
print(df["Release Date"])

0     2012-08-21
1     2026-07-09
2     2023-08-03
3     2024-12-05
4     2013-03-25
         ...    
495   2026-01-27
496   2025-06-17
497   2025-09-08
498   2025-02-13
499   2021-04-08
Name: Release Date, Length: 996, dtype: datetime64[us]


In [366]:
#price
df["Price"] = df["Price"].replace("Free To Play", "0")
df["Price"] = df["Price"].replace(to_replace = r"(?i).*Free.*", value = "0", regex = True)
df["Price"] = df["Price"].replace("No Price", "0")
df["Price"] = df["Price"].str.replace("/ month", "", regex=False)
df["Price"] = df["Price"].replace(to_replace = r"(?i).*demo.*", value = "0", regex = True)
df["Price"] = df["Price"].str.strip("$")
df["Price"] = df["Price"].astype("float")


In [367]:
#discount
df["Discount"] = df["Discount"].replace("No Discount", "0")
df["Discount"] = df["Discount"].str.strip("$")
df["Discount"] = df["Discount"].astype("float")

In [368]:
df["Total Reviews"] = df["Total Reviews"].str.replace(",", "")
df["Total Reviews"] = df["Total Reviews"].astype("int64")

df["Positive Reviews"] = df["Positive Reviews"].str.replace(",", "")
df["Positive Reviews"] = df["Positive Reviews"].astype("int64")

df["Negative Reviews"] = df["Negative Reviews"].str.replace(",", "")
df["Negative Reviews"] = df["Negative Reviews"].astype("int64")

print(df.dtypes)


Name                           str
Release Date        datetime64[us]
Price                      float64
Discount                   float64
Developer                      str
Publisher                      str
Total Reviews                int64
Positive Reviews             int64
Negative Reviews             int64
Review Sentiment               str
Genre(s)                       str
dtype: object


In [369]:
#Separate the genres
import ast
df["Genre(s)"] = df["Genre(s)"].fillna("['None', 'None', 'None', 'None']")
df["Genre(s)"] = df["Genre(s)"].apply(ast.literal_eval)
df["Genre1"] = df["Genre(s)"].str[0]
df["Genre2"] = df["Genre(s)"].str[1]
df["Genre3"] = df["Genre(s)"].str[2]

df.head()


,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s),Genre1,Genre2,Genre3
0,Counter-Strike 2,2012-08-21,0.00,0.00,Valve,Valve,9767142,8401470,1365672,Mostly Positive,"[Action, Free To Play]",Action,Free To Play,NaN
1,Palworld,2026-07-09,29.99,0.00,Pocketpair,Pocketpair,466128,440315,25813,Very Positive,"[Action, Adventure, Indie, RPG]",Action,Adventure,Indie
2,Baldur's Gate 3,2023-08-03,59.99,41.99,Larian Studios,Larian Studios,850775,823465,27310,Very Positive,"[Adventure, RPG, Strategy]",Adventure,RPG,Strategy
3,Marvel Rivals,2024-12-05,0.00,0.00,NetEase Games,NetEase Games,405407,307754,97653,Mixed,"[Action, Free To Play]",Action,Free To Play,NaN
4,Warframe,2013-03-25,1.99,1.59,Digital Extremes,Digital Extremes,678882,593641,85241,Very Positive,"[Action, RPG, Free To Play]",Action,RPG,Free To Play


In [370]:
#Data Cleaning is complete

# Feature Engineering

In [371]:
#Year Released
df["Year Released"] = df["Release Date"].dt.year

In [372]:
#Discount Percentage
df["Discount %"] = round((df["Price"]-df["Discount"])/df["Price"] * 100)
df.head(15)

,Name,Release Date,Price,Discount,Developer,Publisher,Total Reviews,Positive Reviews,Negative Reviews,Review Sentiment,Genre(s),Genre1,Genre2,Genre3,Year Released,Discount %
0,Counter-Strike 2,2012-08-21,0.00,0.00,Valve,Valve,9767142,8401470,1365672,Mostly Positive,"[Action, Free To Play]",Action,Free To Play,NaN,2012,NaN
1,Palworld,2026-07-09,29.99,0.00,Pocketpair,Pocketpair,466128,440315,25813,Very Positive,"[Action, Adventure, Indie, RPG]",Action,Adventure,Indie,2026,100.0
2,Baldur's Gate 3,2023-08-03,59.99,41.99,Larian Studios,Larian Studios,850775,823465,27310,Very Positive,"[Adventure, RPG, Strategy]",Adventure,RPG,Strategy,2023,30.0
3,Marvel Rivals,2024-12-05,0.00,0.00,NetEase Games,NetEase Games,405407,307754,97653,Mixed,"[Action, Free To Play]",Action,Free To Play,NaN,2024,NaN
4,Warframe,2013-03-25,1.99,1.59,Digital Extremes,Digital Extremes,678882,593641,85241,Very Positive,"[Action, RPG, Free To Play]",Action,RPG,Free To Play,2013,20.0
5,Cyberpunk 2077,2020-12-09,59.99,17.99,CD PROJEKT RED,CD PROJEKT RED,973863,840285,133578,Overwhelmingly Positive,[RPG],RPG,NaN,NaN,2020,70.0
6,Assassin's Creed Black Flag Resynced,2026-07-09,59.99,0.00,Ubisoft Singapore,Ubisoft,25624,20307,5317,Very Positive,"[Action, Adventure]",Action,Adventure,NaN,2026,100.0
7,Marvel's Spider-Man 2,2025-01-30,59.99,40.19,Insomniac Games,PlayStation Publishing LLC,42041,34791,7250,Very Positive,"[Action, Adventure]",Action,Adventure,NaN,2025,33.0
8,Halo: Campaign Evolved,2026-07-28,49.99,0.00,Halo Studios,Xbox Game Studios,10155,6494,3661,Mixed,"[Action, Adventure]",Action,Adventure,NaN,2026,100.0
9,Mistfall Hunter,2026-07-29,24.99,22.49,Bellring Games,Skystone Games,5469,3679,1790,Mostly Positive,"[Action, Adventure]",Action,Adventure,NaN,2026,10.0
